## Investigate the sam files of the aligners
### 1. Get general information
- all reads which are tried to be aligned: for bwa result: `/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/notebooks/read_list.txt` 114104932 lines
- check for duplicates: `sort -u /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/notebooks/read_list.txt | uniq -d` => no duplicates in read names

### 2. Get information about the alignment
- how many reads are aligned
- how many reads are not aligned

- bwa_mem2 (broken): `/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/new_alignments/bwa_mem2/error_bwa_mem2_alignment_no_flag.sam`
- bwa: `/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/new_alignments/bwa/bwa_alignment_L80_M_C.sam`
- bowtie: `/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/new_alignments/bowtie/bowtie_alignment_k4_v0.sam`

In [1]:
# imports
import os
import gzip
import pandas as pd

### 1. find all reads

In [2]:
# helpful functions
def get_files(directory):
    files = []
    for file in os.listdir(directory):
        if file.endswith(".fastq.gz"):
            files.append(os.path.join(directory,file))
    return files

def write_all_reads(fastq_dir, out_dir):
    """Write all read names from all fastq files of the given directory into a file. (assumed to be .fastq.gz)"""
    list_files = get_files(fastq_dir)
    # read all fastq files in parallel and store the read names in a list
    read_list = []
    for file in list_files:
        # open gzipped file
        with gzip.open(file, 'rt') as f:
            for line in f:
                if line.startswith('@'):
                    read_list.append(line[1:].strip())

    # store the read list in a file
    with open(os.path.join(out_dir, 'read_list.txt'), 'w') as f:
        for read in read_list:
            f.write(read + '\n')
    return read_list

def print_mapped_unmapped_sequences(aligner, alignment_results_dict):
    """Prints the number of mapped and unmapped sequences and reads for the given aligner."""
    print(f'--------------\nAlignment results for {aligner}:')
    # read alignement specific tsv
    idxstats_tsv = pd.read_csv(alignment_results_dict[aligner], sep='\t', header=None)
    idxstats_tsv.columns = ['seq_name', 'seq_length', 'mapped_reads', 'unmapped_reads']
    # print number of sequences mapped to "*"
    print(f'Number of reads unmapped in {aligner}: {idxstats_tsv[idxstats_tsv["seq_name"] == "*"]["unmapped_reads"].sum()}')
    # remove row with '*' as sequence name
    idxstats_tsv = idxstats_tsv[idxstats_tsv['seq_name'] != '*']
    # remove all reads that have 0 mapped reads
    idxstats_tsv_mapped = idxstats_tsv[idxstats_tsv['mapped_reads'] != 0]
    idxstats_tsv_unmapped = idxstats_tsv[idxstats_tsv['mapped_reads'] == 0]
    if (idxstats_tsv_mapped['seq_name'].is_unique):
        print(f'Number of unique sequences in {aligner}: {idxstats_tsv_mapped["seq_name"].nunique()}')
        # print sum of mapped reads
        print(f'Sum of mapped reads: {idxstats_tsv_mapped["mapped_reads"].sum()}')
    else:
        raise ValueError(f'There are duplicated sequences in the file {aligner}. Not expected.')
    if (idxstats_tsv_unmapped['seq_name'].is_unique):
        print(f'Number of unique unmapped sequences in {aligner}: {idxstats_tsv_unmapped["seq_name"].nunique()}')
    else:
        raise ValueError(f'There are duplicated sequences in the file {aligner}. Not expected.')
    # print sum of mapped and unmapped reads
    print(f'Sum of mapped and unmapped reads: {idxstats_tsv_unmapped["seq_name"].nunique() + idxstats_tsv_mapped["seq_name"].nunique()}')
    print(f'--------------\n')
    return idxstats_tsv_mapped, idxstats_tsv_unmapped

In [10]:
fastq_directory = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTemp/fastq'
fastq_directory_bowtie = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTemp/fastq' 

# get list of all files in directory ending with fastq.gz
output_directory = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/all_reads'
print(f'Writing all reads from {aligner} into a file...')
read_list = write_all_reads(aligner_fastq_dict[aligner], output_directory, aligner)
print(f'Number of reads in {aligner}: {len(read_list)}')
# check sort -u /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/notebooks/read_list.txt | uniq -d
aligner_fastq_dict = {'bwa': fastq_directory_bwa, 'bowtie': fastq_directory_bowtie, 'bowtie2': fastq_directory_bowtie2}



### 2. Find all aligned reads
- Getting number of reads in bam file:  
    - `# get the total number of reads of a BAM file (may include unmapped and duplicated multi-aligned reads)`
        - `samtools view -c SAMPLE.bam`
    - `# counting only mapped (primary aligned) reads`
        - `samtools view -c -F 260 SAMPLE.bam`
- find all mapped reads: `samtools view -b -F 4 file.bam > mapped.bam`
  - bowtie: 103857634 (`samtools view -c -F 260 /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/merged.bam`)
  - bowtie2: 113553518 `samtools view -c -F 260 /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/merged_bowtie2.bam`
  - bwa: 114081455 `samtools view -c -F 260 /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/merged_bwa.bam`

- number of unmapped reads: 
  - bowtie: 10246735 (lines of `samtools view -f 4 /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/merged.bam > /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/unmapped_bowtie.sam`)
  - bowtie2: 550851 (lines of `samtools view -f 4 /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/merged_bowtie2.bam > /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/unmapped_bowtie2.sam`)
  - bwa: 22914 (lines of `samtools view -f 4 /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/merged_bwa.bam > /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/unmapped_bwa.sam`) + (`cat /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/unmapped_bwa.sam | wc -l`)
- go through results from samtools idxstats (how many unique reads are there?)
  - bowtie: `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie.tsv`
  - bowtie2: `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie2.tsv`

In [31]:
# investigate "*" row
aligner = "bwa"
idxstatsasdf_tsv = pd.read_csv(alignment_results_dict[aligner], sep='\t', header=None)
idxstatsasdf_tsv

,0,1,2,3
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,639,0
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,441,0
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,206,0
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,164,0
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,723,0
...,...,...,...,...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,300,88,0
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,300,2647,0
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,300,431,0
80214,MK:tile_22033|chr2-71506006+71506275|scramble_...,300,0,0


In [5]:
# input dict:
# {aligner_1: input_tsv, aligner_2: input_tsv, ...}
alignment_results_dict = {
    'bwa': '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/idxstats_bwa.tsv',
    'bowtie': '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie.tsv',
    'bowtie2': '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie2.tsv',
    'bowtie2_mutliple': '/data/gpfs-1/groups/ag_kircher/work/MPRA/IGVF_Y1_design/experiment/results/multiple_alignments/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie2.tsv',
    'bwa_multiple': '/data/gpfs-1/groups/ag_kircher/work/MPRA/IGVF_Y1_design/experiment/standard_results/results/all_alignments/standardAssignIGVFDesignNoTemp/bam/idxstats_bwa_multiple.tsv',
    'bwa-mem2': '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/bwa2/assignment/standardAssignIGVFDesignNoTemp/bam/idxstats_bwa-mem2.tsv',
    'bwa_1201': '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/idxstats_bwa_1201.tsv',
    'bwa_view_1792': '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/idxstats_bwa_merged_view_1792_output.tsv',
}

for key, value in alignment_results_dict.items():
    idxstats_mapped_df, idxstats_unmapped_df = print_mapped_unmapped_sequences(key, alignment_results_dict)

--------------
Alignment results for bwa:
Number of reads unmapped in bwa: 22914
Number of unique sequences in bwa: 79291
Sum of mapped reads: 114096050
Number of unique unmapped sequences in bwa: 924
Sum of mapped and unmapped reads: 80215
--------------

--------------
Alignment results for bowtie:
Number of reads unmapped in bowtie: 10246735
Number of unique sequences in bowtie: 78613
Sum of mapped reads: 103857634
Number of unique unmapped sequences in bowtie: 1602
Sum of mapped and unmapped reads: 80215
--------------

--------------
Alignment results for bowtie2:
Number of reads unmapped in bowtie2: 550851
Number of unique sequences in bowtie2: 79092
Sum of mapped reads: 113553518
Number of unique unmapped sequences in bowtie2: 1123
Sum of mapped and unmapped reads: 80215
--------------

--------------
Alignment results for bowtie2_mutliple:
Number of reads unmapped in bowtie2_mutliple: 550414
Number of unique sequences in bowtie2_mutliple: 79586
Sum of mapped reads: 328845518
Nu

### Use the awk command
global:
  assignments:
    split_number: 30
assignments:
  standardAssignIGVFDesignNoTemp:
    bc_length: 15
    sequence_length:
      min: 265
      max: 275
    alignment_start:
      min: 15
      max: 17
    min_mapping_quality: 1
awk -v "OFS=\t" '{{
            split($(NF),a,":");
            split(a[3],a,",");
            if (a[1] !~ /N/) {{
                if (($5 >= 1) && ($4 >= 15) && ($4 <= 17) && (length($10) >= 265) && (length($10) <= 275)) {{
                    print a[1],$3,$4";"$6";"$12";"$13";"$5 
                }} else {{
                    print a[1],"other","NA" 
                }}
            }}
        }}' | sort -k1,1 -k2,2 -k3,3 > barcodes_incl_other.bwa_1201.tsv

### 3. Find all unaligned reads

In [ ]:
# command line command to sort a row of a file and check if duplicates exist
# sort -u read_list_bwa.txt | uniq -d